# Allocation validation against naive benchmarks

**Purpose.** Test whether the RF-driven mean-variance optimiser adds value beyond a naive 1/N (equal-weighting) baseline on the held-out window (2024-01-01 onwards). The 1/N benchmark follows DeMiguel, Garlappi and Uppal (2009), *Optimal Versus Naive Diversification: How Inefficient is the 1/N Portfolio Strategy?*, the standard test of whether mean-variance optimisation overcomes its known estimation-error fragility.

**Methodology.**

- **Fixed stock selection** of 15 sector-diverse FTSE 100 stocks: HSBA, BARC, LLOY, BP, SHEL, ULVR, DGE, TSCO, AZN, GSK, RIO, GLEN, VOD, NG, PRU. Holding the selection fixed isolates the *allocation* question from the *stock-picking* question, both 1/N and the optimiser work with the same universe.
- **Quarterly rebalancing** at the last trading day of each calendar quarter in the held-out window. At each rebalance date, the optimiser is re-run using only data available up to that date (predictions + Ledoit-Wolf correlation strictly leakage-free).
- **Three strategies** evaluated head-to-head:
  1. *Optimised (Balanced)*: SLSQP mean-variance with RF-predicted vols on the diagonal, Ledoit-Wolf correlation off-diagonal, shrinkage α=0.75, λ=2.0.
  2. *Equal weighting (1/N)*: 1/15 across all 15 stocks each quarter.
  3. *FTSE 100*: buy-and-hold the index over the same period.
- **Realised quarterly returns** computed from the next 63 trading days following each rebalance.
- **Summary metrics**: cumulative return, annualised volatility, Sharpe ratio (risk-free rate = 0), maximum drawdown.

**Models.** rf_return_v6 and rf_volatility_v6, trained on data ending 2023-12-31. The entire backtest window is out-of-sample.

In [1]:
import sys, warnings
warnings.filterwarnings('ignore')
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import joblib
from pathlib import Path
import plotly.graph_objects as go
from IPython.display import display

from src.models.optimise import optimise_portfolio
from lib.bands import BANDS, effective_cap

## 1. Load data and define the test selection

In [ ]:
features = pd.read_parquet('../data/processed/features.parquet')
prices_clean = pd.read_parquet('../data/processed/prices_clean.parquet')
rf_return = joblib.load('../models/rf_return_v6.joblib')
rf_volatility = joblib.load('../models/rf_volatility_v6.joblib')
feature_cols = list(rf_return.feature_names_in_)

selected = ['HSBA.L', 'BARC.L', 'LLOY.L', 'BP.L', 'SHEL.L',
            'ULVR.L', 'DGE.L', 'TSCO.L', 'AZN.L', 'GSK.L',
            'RIO.L', 'GLEN.L', 'VOD.L', 'NG.L', 'PRU.L']
print(f'Selection size: {len(selected)} stocks')
print(f'Effective per-asset cap: {effective_cap(len(selected)) * 100:.1f}%')
print(f'Feature columns: {feature_cols}')

## 2. Identify rebalance dates

A valid rebalance date `T` must satisfy two conditions:
1. All selected tickers have non-NaN features at `T` (none of the selected stocks are late listings inside the held-out window).
2. The realised forward return at `T` is computable, i.e., `T + 63` trading days exists in the price panel.

The second condition naturally truncates the last quarter or so of the price panel.

In [ ]:
features_dates = features.index.get_level_values('date').unique().sort_values()
held_out_dates = features_dates[features_dates >= pd.Timestamp('2024-01-01')]

# Last trading day of each calendar quarter in the held-out window
quarters_df = pd.DataFrame({'date': held_out_dates})
quarters_df['yq'] = quarters_df['date'].dt.to_period('Q')
quarter_ends = quarters_df.groupby('yq')['date'].max().sort_values().tolist()

valid_quarter_ends = []
for T in quarter_ends:
    try:
        cross = features.xs(T, level='date')
    except KeyError:
        continue
    if not all(t in cross.index for t in selected):
        continue
    if cross.loc[selected, 'forward_return'].isna().any():
        continue
    valid_quarter_ends.append(T)

print(f'Valid rebalance dates: {len(valid_quarter_ends)}')
print(f'First: {valid_quarter_ends[0].date()}')
print(f'Last:  {valid_quarter_ends[-1].date()}')

## 3. Rolling backtest

For each rebalance date `T`, run the optimiser using features at `T` (out-of-sample for any T in 2024-01 onwards) and historical returns strictly up to `T`. Hold the resulting weights for the next 63 trading days and record the realised portfolio return.

In [ ]:
config = BANDS['Balanced']
cap = effective_cap(len(selected))

records = []
for T in valid_quarter_ends:
    cross = features.xs(T, level='date')
    X = cross.loc[selected, feature_cols]
    mu = pd.Series(rf_return.predict(X), index=selected)
    sigma = pd.Series(rf_volatility.predict(X), index=selected)

    # Historical returns strictly up to T (no leakage into Ledoit-Wolf)
    prices_close = prices_clean['Close'][selected].loc[:T]
    daily_returns = prices_close.pct_change(fill_method=None).dropna(how='all')

    result = optimise_portfolio(
        mu, sigma, daily_returns,
        risk_aversion=config.risk_aversion,
        shrinkage_alpha=config.shrinkage_alpha,
        max_weight=cap,
    )
    optimised_weights = result['weights'].reindex(selected).fillna(0.0)

    realised = cross.loc[selected, 'forward_return']
    opt_return = float((optimised_weights * realised).sum())
    eq_return = float(realised.mean())   # 1/N: weights are 1/15 per stock

    # FTSE 100 over the same 63-trading-day forward window
    ftse_prices = prices_clean['Close']['^FTSE']
    ftse_dates = ftse_prices.index
    idx_T = ftse_dates.get_loc(T)
    if idx_T + 63 < len(ftse_dates):
        T_plus_63 = ftse_dates[idx_T + 63]
        ftse_return = float(ftse_prices.loc[T_plus_63] / ftse_prices.loc[T] - 1)
    else:
        ftse_return = np.nan

    records.append({
        'date': T,
        'optimised': opt_return,
        'equal_weight': eq_return,
        'ftse_100': ftse_return,
    })

returns_df = pd.DataFrame(records).set_index('date').dropna()
print(f'{len(returns_df)} quarterly returns recorded')
print()
display(returns_df.head())
display(returns_df.tail())

## 4. Cumulative paths and summary metrics

In [ ]:
cum = (1 + returns_df).cumprod()

def summary_metrics(quarterly: pd.Series) -> dict:
    cumret = (1 + quarterly).prod() - 1
    quarterly_std = quarterly.std()
    annvol = quarterly_std * np.sqrt(4)
    sharpe = (quarterly.mean() * 4) / annvol if annvol > 0 else np.nan
    cum_ = (1 + quarterly).cumprod()
    drawdown = float((cum_ / cum_.cummax() - 1).min())
    return {
        'Cumulative return': cumret,
        'Annualised vol':    annvol,
        'Sharpe ratio':      sharpe,
        'Max drawdown':      drawdown,
    }

summary = pd.DataFrame({col: summary_metrics(returns_df[col]) for col in returns_df.columns}).T

summary_display = summary.copy()
summary_display['Cumulative return'] = summary_display['Cumulative return'].apply(lambda x: f'{x*100:+.2f}%')
summary_display['Annualised vol'] = summary_display['Annualised vol'].apply(lambda x: f'{x*100:.2f}%')
summary_display['Sharpe ratio'] = summary_display['Sharpe ratio'].apply(lambda x: f'{x:.2f}')
summary_display['Max drawdown'] = summary_display['Max drawdown'].apply(lambda x: f'{x*100:.2f}%')
summary_display.index = ['Optimised (Balanced)', 'Equal weight (1/N)', 'FTSE 100']

print(f'Backtest window: {returns_df.index[0].date()} to {returns_df.index[-1].date()}')
print(f'Number of quarterly observations: {len(returns_df)}')
print()
display(summary_display)

## 5. Cumulative return chart

In [ ]:
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=cum.index, y=cum['optimised'],
    mode='lines', name='Optimised (Balanced)',
    line=dict(color='#0F2540', width=2.8),
    hovertemplate='%{x|%Y-%m-%d}<br>Optimised: %{y:.3f}<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=cum.index, y=cum['equal_weight'],
    mode='lines', name='Equal weight (1/N)',
    line=dict(color='#0E8E8E', width=2.2, dash='dash'),
    hovertemplate='%{x|%Y-%m-%d}<br>1/N: %{y:.3f}<extra></extra>',
))
fig.add_trace(go.Scatter(
    x=cum.index, y=cum['ftse_100'],
    mode='lines', name='FTSE 100',
    line=dict(color='#5A5A5A', width=2, dash='dot'),
    hovertemplate='%{x|%Y-%m-%d}<br>FTSE 100: %{y:.3f}<extra></extra>',
))

fig.add_hline(y=1.0, line=dict(color='#5A5A5A', width=1, dash='dot'), opacity=0.4)

fig.update_layout(
    height=440,
    margin=dict(l=20, r=20, t=40, b=40),
    plot_bgcolor='#FFFFFF',
    paper_bgcolor='#FFFFFF',
    title=dict(
        text=f'Cumulative return, held-out backtest ({returns_df.index[0].date()} to {returns_df.index[-1].date()})',
        font=dict(size=14, color='#1A1A1A'),
        x=0,
    ),
    xaxis=dict(title='Rebalance date', color='#5A5A5A', showgrid=False),
    yaxis=dict(
        title='Cumulative return (multiplier on £1 invested)',
        color='#5A5A5A', gridcolor='#E4E2DC',
    ),
    legend=dict(
        orientation='h', yanchor='bottom', y=1.02,
        xanchor='right', x=1, bgcolor='rgba(255,255,255,0)',
    ),
    hovermode='x unified',
)
fig.show()

## 6. Findings and discussion

### 6.1 Headline result

Over **[N]** quarterly rebalances from **[FIRST_DATE]** to **[LAST_DATE]**, on a fixed selection of 15 sector-diverse FTSE 100 stocks:

| Strategy             | Cumulative return | Annualised vol | Sharpe ratio | Max drawdown |
|----------------------|-------------------|----------------|--------------|--------------|
| Optimised (Balanced) | [TBD]             | [TBD]          | [TBD]        | [TBD]        |
| Equal weight (1/N)   | [TBD]             | [TBD]          | [TBD]        | [TBD]        |
| FTSE 100             | [TBD]             | [TBD]          | [TBD]        | [TBD]        |

*Numbers will be filled in from the cell 10 (`summary_display`) output above after re-running. The interpretation framework below is generic and applies whether or not 1/N still dominates at the quarterly horizon, only the specific numerical claims need updating.*

### 6.2 Why the result may differ from v5 monthly

The v5 monthly backtest (`docs/methodology.md` §4.5, reproduced in commit history) found 1/N dominating on every metric, consistent with DeMiguel, Garlappi and Uppal (2009). Two effects could shift the v6 quarterly result:

- **Stronger return signal at quarterly horizon.** The Return RF v6's held-out Spearman ρ (0.088) is 4.4× v5's (0.020). A stronger rank signal feeding into the mean-variance objective should produce more discriminating allocations. *Whether* this translates into beating 1/N is the empirical question this backtest answers.
- **Implicit risk-aversion shift.** The unit mismatch in the optimiser objective (annual variance vs per-horizon return) means that at the same band λ values, the v6 optimiser is ~3× less effectively risk-averse than v5 was. This was a documented intentional shift at Step 4 of the refactor, the assumption being that stronger forward signal warrants greater weight in allocation. The backtest is the truth-check on whether this shift was helpful.

### 6.3 Methodological interpretation framework

Regardless of which way the numbers land, the project's core position is that the contribution lies in the *trust-intervention layers* rather than numerical alpha generation:

1. **Per-stock SHAP attributions** that explain why each ticker received its predicted return and volatility.
2. **Analytical weight decomposition** that explains why the optimiser arrived at each ticker's allocation, what the return signal contributed, what individual volatility contributed, what diversification contributed, and what the user's risk profile contributed.
3. **Constrained-agency modification** (Dietvorst, Simmons and Massey, 2018), the ±5pp slider that lets users put their fingerprint on the recommendation while preserving the optimiser's overall risk-management work.
4. **Risk-band differentiation**, the four bands (Cautious, Balanced, Growth, Adventurous) deliver portfolios that respond to user preference; 1/N is risk-band-agnostic by construction.

If the optimised allocation now beats 1/N at quarterly horizon, the case for the prediction-and-optimisation layer strengthens. If 1/N continues to dominate, the case for the trust-intervention layers remains unchanged (as documented in v5's discussion).

### 6.4 Caveats (unchanged from v5)

- **Single selection.** The backtest uses one fixed 15-stock universe. A robustness check across multiple selections (different sectors, different sizes) would strengthen the conclusion.
- **Short evaluation window.** Quarterly rebalancing in 2024-01 → 2026-02 yields ~9 observations, even more statistically modest than the 28 monthly observations of v5. The Sharpe-ratio differences in particular are subject to substantial sampling variation.
- **Frictionless rebalancing.** No transaction costs, no bid-ask spreads, no slippage. Quarterly rebalancing reduces turnover materially vs monthly, the optimised allocation benefits proportionally more from the frictionless assumption at this lower cadence.
- **No retraining.** The RF models were trained once on pre-2024 data and held fixed for the full backtest. A walk-forward retraining schedule is future work.

### 6.5 For the dissertation

Suggested template wording (fill in numbers from the actual run):

> *Following DeMiguel, Garlappi and Uppal (2009), the optimised mean-variance allocation was tested against a naive equal-weighting (1/N) benchmark and the FTSE 100 index on a held-out quarterly backtest from [FIRST_DATE] to [LAST_DATE] (n = [N] rebalance observations). [Insert summary statement based on results.] [If 1/N still dominates: "Result consistent with the well-documented failure mode of mean-variance optimisation under noisy return inputs."] [If optimiser improves materially: "Result suggests the stronger quarterly return signal partially overcomes the estimation-error fragility documented by DeMiguel et al. (2009)."]. The project's framing, contribution lies in the explainability and constrained-agency layers, remains valid in either case.*

### References

DeMiguel, V., Garlappi, L. and Uppal, R. (2009) 'Optimal versus naive diversification: How inefficient is the 1/N portfolio strategy?', *The Review of Financial Studies*, 22(5), pp. 1915–1953.

## 7. Step 5b, λ sensitivity (Balanced band)

The v6 quarterly backtest (Section 4 above) showed the optimised portfolio losing the vol-reduction property it had under v5, annualised vol came in slightly above 1/N. Because μ is per-horizon while Σ is annualised, the unit mismatch is absorbed by the risk-aversion parameter λ. This section re-runs the same rolling backtest at three values of λ, the current default (2), plus 4 and 6, to test whether re-tuning the Balanced band recovers the expected behaviour, or whether the gap is sample noise (9 quarterly observations).

Nothing else changes: same selection, same quarter-ends, same α = 0.75, same cap, same v6 model artifacts.


In [ ]:
def run_backtest(risk_aversion: float, shrinkage_alpha: float = 0.75) -> pd.DataFrame:
    """Re-run the Section 3 rolling backtest at a given lambda.

    Returns a DataFrame indexed by rebalance date with columns:
    'optimised', 'equal_weight', 'ftse_100'. The 1/N and FTSE legs do
    not depend on lambda but are recomputed for self-contained output.
    """
    cap_ = effective_cap(len(selected))
    rows = []
    for T in valid_quarter_ends:
        cross = features.xs(T, level='date')
        X = cross.loc[selected, feature_cols]
        mu = pd.Series(rf_return.predict(X), index=selected)
        sigma = pd.Series(rf_volatility.predict(X), index=selected)

        prices_close = prices_clean['Close'][selected].loc[:T]
        daily_returns = prices_close.pct_change(fill_method=None).dropna(how='all')

        result = optimise_portfolio(
            mu, sigma, daily_returns,
            risk_aversion=risk_aversion,
            shrinkage_alpha=shrinkage_alpha,
            max_weight=cap_,
        )
        w_opt = result['weights'].reindex(selected).fillna(0.0)
        realised = cross.loc[selected, 'forward_return']

        ftse_prices = prices_clean['Close']['^FTSE']
        ftse_dates = ftse_prices.index
        idx_T = ftse_dates.get_loc(T)
        if idx_T + 63 < len(ftse_dates):
            T_plus_63 = ftse_dates[idx_T + 63]
            ftse_ret = float(ftse_prices.loc[T_plus_63] / ftse_prices.loc[T] - 1)
        else:
            ftse_ret = np.nan

        rows.append({
            'date': T,
            'optimised':    float((w_opt * realised).sum()),
            'equal_weight': float(realised.mean()),
            'ftse_100':     ftse_ret,
        })
    return pd.DataFrame(rows).set_index('date').dropna()


print('run_backtest() defined.')


In [ ]:
lambdas = [2.0, 4.0, 6.0]
runs = {}
for lam in lambdas:
    print(f'Running backtest at lambda = {lam} ...')
    runs[lam] = run_backtest(risk_aversion=lam, shrinkage_alpha=0.75)

# Build comparison table: one row per lambda for the optimised leg.
rows = []
for lam in lambdas:
    m = summary_metrics(runs[lam]['optimised'])
    rows.append({
        'lambda':            lam,
        'Cumulative return': m['Cumulative return'],
        'Annualised vol':    m['Annualised vol'],
        'Sharpe ratio':      m['Sharpe ratio'],
        'Max drawdown':      m['Max drawdown'],
    })

# Reference rows: 1/N and FTSE (taken from any one run, they don't depend on lambda)
ref = runs[lambdas[0]]
for label, col in [('1/N', 'equal_weight'), ('FTSE 100', 'ftse_100')]:
    m = summary_metrics(ref[col])
    rows.append({
        'lambda':            label,
        'Cumulative return': m['Cumulative return'],
        'Annualised vol':    m['Annualised vol'],
        'Sharpe ratio':      m['Sharpe ratio'],
        'Max drawdown':      m['Max drawdown'],
    })

comparison = pd.DataFrame(rows).set_index('lambda')
comparison_display = comparison.copy()
comparison_display['Cumulative return'] = comparison_display['Cumulative return'].apply(lambda x: f'{x*100:+.2f}%')
comparison_display['Annualised vol'] = comparison_display['Annualised vol'].apply(lambda x: f'{x*100:.2f}%')
comparison_display['Sharpe ratio'] = comparison_display['Sharpe ratio'].apply(lambda x: f'{x:.2f}')
comparison_display['Max drawdown'] = comparison_display['Max drawdown'].apply(lambda x: f'{x*100:.2f}%')

print()
print(f'Backtest window: {ref.index[0].date()} to {ref.index[-1].date()} ({len(ref)} quarters)')
display(comparison_display)


### Verdict

| λ | Annualised vol | Sharpe |
|---|---|---|
| 2 | 10.43% | 1.13 |
| 4 | 10.52% | 1.12 |
| 6 | 10.56% | 1.11 |

Vol moves by 0.13pp across the full λ range; Sharpe by 0.02, an order of magnitude below the 1pp decision threshold. λ is effectively inert under the v6 setup (per-horizon μ, annual Σ, 10% cap, 15 stocks, α = 0.75).

**Decision: accept the v6 backtest as-is and keep the Balanced band at λ = 2.** The 0.71pp gap between v5's 9.72% optimised vol and v6's 10.43% sits comfortably inside the sampling variability of a 9-quarter window. The v5-vs-v6 difference is not a λ-tuning artefact and not a recoverable unit-mismatch issue at this sample size, re-tuning λ does not move the needle.


## 8. Step 5c, shrinkage (α) sensitivity (Balanced band)

The Step 5b λ sweep showed risk-aversion has effectively no influence under the v6 setup. But there's a second lever that controls how much of the RF's per-stock signal flows through to the optimiser: the shrinkage parameter α. With α = 0.75 (current Balanced default), 75% of each stock's RF prediction is preserved and 25% is pulled toward the cross-sectional mean, a heavy assumption that the RF's stock-picking signal is noisy. Under a quarterly horizon, that signal may be more robust than under monthly (less per-observation noise), so a lower α (more trust in the RF) might be defensible.

This section re-runs the backtest at α ∈ {0.75 (current), 0.5, 0.35} while holding λ = 2 fixed. Everything else (selection, quarter-ends, cap, v6 model artifacts) stays unchanged.


In [ ]:
alphas = [0.75, 0.5, 0.35]
alpha_runs = {}
for alpha in alphas:
    print(f'Running backtest at alpha = {alpha} (lambda = 2.0) ...')
    alpha_runs[alpha] = run_backtest(risk_aversion=2.0, shrinkage_alpha=alpha)

rows = []
for alpha in alphas:
    m = summary_metrics(alpha_runs[alpha]['optimised'])
    rows.append({
        'alpha':             alpha,
        'Cumulative return': m['Cumulative return'],
        'Annualised vol':    m['Annualised vol'],
        'Sharpe ratio':      m['Sharpe ratio'],
        'Max drawdown':      m['Max drawdown'],
    })

# Reference rows
ref = alpha_runs[alphas[0]]
for label, col in [('1/N', 'equal_weight'), ('FTSE 100', 'ftse_100')]:
    m = summary_metrics(ref[col])
    rows.append({
        'alpha':             label,
        'Cumulative return': m['Cumulative return'],
        'Annualised vol':    m['Annualised vol'],
        'Sharpe ratio':      m['Sharpe ratio'],
        'Max drawdown':      m['Max drawdown'],
    })

alpha_comparison = pd.DataFrame(rows).set_index('alpha')
alpha_display = alpha_comparison.copy()
alpha_display['Cumulative return'] = alpha_display['Cumulative return'].apply(lambda x: f'{x*100:+.2f}%')
alpha_display['Annualised vol'] = alpha_display['Annualised vol'].apply(lambda x: f'{x*100:.2f}%')
alpha_display['Sharpe ratio'] = alpha_display['Sharpe ratio'].apply(lambda x: f'{x:.2f}')
alpha_display['Max drawdown'] = alpha_display['Max drawdown'].apply(lambda x: f'{x*100:.2f}%')

print()
print(f'Backtest window: {ref.index[0].date()} to {ref.index[-1].date()} ({len(ref)} quarters)')
display(alpha_display)


In [ ]:
# Diagnostic: portfolio expected return at the most recent rebalance date,
# for each alpha. This is the value that flows into project_forward's
# period_return parameter on the Forward Projection page. Translates the
# academic backtest above into the user-facing cone behaviour.
from src.models.optimise import optimise_portfolio

T_latest = valid_quarter_ends[-1]
cross_latest = features.xs(T_latest, level='date')
X_latest = cross_latest.loc[selected, feature_cols]
mu_latest = pd.Series(rf_return.predict(X_latest), index=selected)
sigma_latest = pd.Series(rf_volatility.predict(X_latest), index=selected)
prices_T = prices_clean['Close'][selected].loc[:T_latest]
daily_T = prices_T.pct_change(fill_method=None).dropna(how='all')

diag_rows = []
for alpha in alphas:
    result = optimise_portfolio(
        mu_latest, sigma_latest, daily_T,
        risk_aversion=2.0,
        shrinkage_alpha=alpha,
        max_weight=effective_cap(len(selected)),
    )
    er_q = float(result['expected_return'])
    er_ann = (1 + er_q) ** 4 - 1
    diag_rows.append({
        'alpha':                  alpha,
        'Portfolio E[r] (1q)':    f'{er_q*100:+.2f}%',
        'Portfolio E[r] (1yr)':   f'{er_ann*100:+.2f}%',
        'Expected vol (annual)':  f'{float(result["expected_vol"])*100:.2f}%',
    })

diag = pd.DataFrame(diag_rows).set_index('alpha')
print(f'Cross-section date: {T_latest.date()}')
print('Translates to the projection-cone median on the Forward Projection page.')
display(diag)


### Verdict

**Backtest sweep (α at fixed λ = 2):**

| α | Cumulative return | Annualised vol | Sharpe |
|---|---|---|---|
| 0.75 | +26.99% | 10.73% | 1.05 |
| 0.50 | +27.34% | 10.61% | 1.07 |
| 0.35 | +27.52% | 10.54% | 1.08 |

Movement across full α range: +0.53pp return, −0.19pp vol, +0.03 Sharpe. Well below the 3pp / 1pp decision threshold.

**Cross-section diagnostic at the latest rebalance date:** Portfolio E[r] is +2.14% quarterly (+8.86% annualised) at every α tested, *identical to four decimal places*. Annualised vol is ~12.7% across all α.

**Decision: accept α = 0.75. No change to the Balanced band default.** α is mathematically inert under the v6 constraint configuration. The 10% per-asset cap on a 15-stock selection forces near-uniform weights; under uniform weights, the portfolio expected return collapses to the cross-sectional mean of the RF's per-stock predictions, regardless of α:

```
w @ (α·μ + (1-α)·μ̄) ≈ α·μ̄ + (1-α)·μ̄ = μ̄   when w is uniform.
```

Shrinkage would only have leverage if the cap were looser, allowing concentrated positions on high-μ stocks. Under the current safety-first cap policy (Brown & Goetzmann, 1995, on the dangers of concentration), shrinkage is dormant, which has the side benefit that the band default is robust to shrinkage misspecification.

**Implication for the projection cone:** The £1,016 / +1.6% projection observed on a specific user-built portfolio is a *selection effect*, not a band-default issue. The test selection used in this notebook (a broadly diversified 15-stock FTSE mix) gives portfolio E[r] of +8.86% annualised, competitive with FTSE's +8.1%. A portfolio dominated by stocks with low RF-predicted quarterly returns will produce a much weaker cone. This is honest model output, not a tuning problem.


## 9. Pipeline ablation, RF-vol vs historical-vol diagonal

The sensitivity sweeps in §7 and §8 establish *internal* robustness of the band defaults. This section asks a different question: does the hybrid pipeline's RF-predicted volatility diagonal earn its place against the simpler alternative of a Ledoit-Wolf-shrunk covariance built from *historical* realised volatilities?

Two configurations on the same backtest infrastructure, Balanced band defaults (λ=2.0, α=0.75):
- **Pipeline**: Σ = D_RF · C_LW · D_RF, the dissertation's hybrid construction.
- **Baseline**: Σ = D_hist · C_LW · D_hist, same Ledoit-Wolf correlation; diagonal replaced by 252-day trailing realised volatility, annualised.

The optimiser, return predictor (μ from rf_return_v6), risk-band parameters, cap policy, and benchmarks (1/N, FTSE 100) are held constant.


In [ ]:
def run_backtest_hist_vol(risk_aversion: float = 2.0,
                          shrinkage_alpha: float = 0.75) -> pd.DataFrame:
    """Same as run_backtest() but replaces the RF-predicted volatility
    vector with 252-day trailing realised volatility, annualised."""
    cap_ = effective_cap(len(selected))
    rows = []
    for T in valid_quarter_ends:
        cross = features.xs(T, level='date')
        X = cross.loc[selected, feature_cols]
        mu = pd.Series(rf_return.predict(X), index=selected)

        prices_close = prices_clean['Close'][selected].loc[:T]
        daily_returns = prices_close.pct_change(fill_method=None).dropna(how='all')

        # Diagonal swap: historical realised vol, annualised
        sigma_hist = (daily_returns.tail(252).std() * np.sqrt(252)).reindex(selected)

        result = optimise_portfolio(
            mu, sigma_hist, daily_returns,
            risk_aversion=risk_aversion,
            shrinkage_alpha=shrinkage_alpha,
            max_weight=cap_,
        )
        w = result['weights'].reindex(selected).fillna(0.0)
        realised = cross.loc[selected, 'forward_return']

        ftse_prices = prices_clean['Close']['^FTSE']
        ftse_dates = ftse_prices.index
        idx_T = ftse_dates.get_loc(T)
        if idx_T + 63 < len(ftse_dates):
            ftse_ret = float(ftse_prices.iloc[idx_T + 63] / ftse_prices.loc[T] - 1)
        else:
            ftse_ret = np.nan

        rows.append({
            'date': T,
            'optimised':    float((w * realised).sum()),
            'equal_weight': float(realised.mean()),
            'ftse_100':     ftse_ret,
        })
    return pd.DataFrame(rows).set_index('date').dropna()

print('Running RF-vol pipeline (re-run for direct comparison) ...')
rf_run    = run_backtest(risk_aversion=2.0, shrinkage_alpha=0.75)
print('Running hist-vol baseline ...')
hist_run  = run_backtest_hist_vol(risk_aversion=2.0, shrinkage_alpha=0.75)
print(f'Both runs completed across {len(rf_run)} quarterly observations.')


In [ ]:
rows = []
for label, series in [
    ('RF-vol x LW-corr (pipeline)',  rf_run['optimised']),
    ('hist-vol x LW-corr (baseline)', hist_run['optimised']),
    ('1/N',                          rf_run['equal_weight']),
    ('FTSE 100',                     rf_run['ftse_100']),
]:
    m = summary_metrics(series)
    rows.append({
        'Configuration':     label,
        'Cumulative return': m['Cumulative return'],
        'Annualised vol':    m['Annualised vol'],
        'Sharpe ratio':      m['Sharpe ratio'],
        'Max drawdown':      m['Max drawdown'],
    })

ablation_df = pd.DataFrame(rows).set_index('Configuration')
display_df = ablation_df.copy()
display_df['Cumulative return'] = display_df['Cumulative return'].apply(lambda x: f'{x*100:+.2f}%')
display_df['Annualised vol']    = display_df['Annualised vol'].apply(lambda x: f'{x*100:.2f}%')
display_df['Sharpe ratio']      = display_df['Sharpe ratio'].apply(lambda x: f'{x:.2f}')
display_df['Max drawdown']      = display_df['Max drawdown'].apply(lambda x: f'{x*100:.2f}%')

print(f'Backtest window: {rf_run.index[0].date()} -> {rf_run.index[-1].date()} ({len(rf_run)} quarters)')
display(display_df)

# Pipeline minus baseline (the empirical core of the ablation)
m_rf   = summary_metrics(rf_run['optimised'])
m_hist = summary_metrics(hist_run['optimised'])
print()
print('Pipeline - baseline:')
print(f"  Cumulative return: {(m_rf['Cumulative return'] - m_hist['Cumulative return'])*100:+.2f}pp")
print(f"  Annualised vol:    {(m_rf['Annualised vol']    - m_hist['Annualised vol'])*100:+.2f}pp")
print(f"  Sharpe ratio:      {m_rf['Sharpe ratio'] - m_hist['Sharpe ratio']:+.3f}")
print(f"  Max drawdown:      {(m_rf['Max drawdown']      - m_hist['Max drawdown'])*100:+.2f}pp")


### Verdict

| Configuration | Cumulative return | Annualised vol | Sharpe | Max drawdown |
|---|---|---|---|---|
| **RF-vol × LW-corr (pipeline)** | **+31.08%** | **10.43%** | **1.36** | −4.50% |
| hist-vol × LW-corr (baseline) | +22.42% | 8.74% | 1.21 | −4.64% |
| 1/N equal-weight | +35.80% | 9.82% | 1.63 | −3.77% |
| FTSE 100 | +25.64% | 4.51% | 2.58 | −1.06% |

**Pipeline − baseline:** +8.66pp cumulative return, +0.15 Sharpe, +1.70pp vol, +0.14pp max drawdown.

The RF-predicted volatility diagonal materially improves realised return-per-unit-of-risk over the simpler historical-volatility baseline across the eight-quarter held-out window. The direction supports the architectural claim that the RF-vol contribution is doing measurable work rather than being decorative.

Two honest caveats are reported in §5.7 of `docs/methodology.md`:
1. Equal-weighted 1/N continues to outperform both optimised configurations on Sharpe (DeMiguel, Garlappi & Uppal, 2009, mean-variance optimisation is hard to beat 1/N out-of-sample at small portfolio sizes); the dissertation does not claim the optimised pipeline beats 1/N, only that the hybrid composition beats its closest simpler alternative.
2. The FTSE 100's Sharpe of 2.58 is a small-sample artefact (annualised vol estimated at 4.51% over only 8 quarters, implausibly low for the FTSE 100); not a meaningful benchmark for Sharpe comparisons at this sample size.
